# 1. Téléchargements des données et packages requis

Dans cette première étape, nous récupérons le jeu de données **APPA-REAL** depuis le site officiel.  
Les commandes ci-dessous permettent de télécharger l’archive contenant les images et les annotations, puis de la décompresser afin de rendre les fichiers accessibles pour les étapes suivantes de l’analyse et du prétraitement.

In [ ]:
!wget https://data.chalearnlap.cvc.uab.cat/AppaRealAge/appa-real-release.zip
!unzip appa-real-release.zip

In [1]:
requirements = """torch
torchvision
torchaudio

numpy==1.26.4
pandas
matplotlib
opencv-python
Pillow
scikit-learn

pretrainedmodels
albumentations
imgaug
tqdm

tensorboard
yacs
better-exceptions
timm>=0.9.16
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)
!pip install -r requirements.txt


In [3]:
import torch
torch.cuda.is_available()

True

## 1.2 Modèle de base pour l'estimation de l'âge

Dans un premier temps, nous utilisons un **modèle de base pour l’estimation de l’âge** basé sur une approche de **classification discrète**. L’âge est représenté comme un problème de classification sur **101 classes (0 à 100 ans)**. Le réseau est entraîné avec une **fonction de perte Cross-Entropy**, qui encourage le modèle à prédire correctement la classe correspondant à l’âge réel.

Lors de l’inférence, la prédiction finale de l’âge est obtenue simplement en prenant **la classe ayant la probabilité maximale** (`argmax`) parmi les sorties du réseau. Autrement dit, l’âge estimé correspond directement à la classe prédite la plus probable.

Cette version, appelée **mode `none` dans le code**, constitue une **baseline simple** sur laquelle nous chercherons ensuite à apporter des améliorations (par exemple avec des méthodes comme DEX, residual learning ou des modèles probabilistes).

In [ ]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD none

3.889

![Texte alternatif](Images/training_curves_none02.png)

In [ ]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/last.pth MODEL.METHOD none

![Texte alternatif](Images/noneTest.png)

La valeur à améliorer est 5.191 donc en moyenne le modèle se trompe de 5 ans environ.

## LabelSmoothing & Dropout

### LabelSmoothing


Le *label smoothing* est une technique utilisée pour améliorer la généralisation d’un modèle de classification.  
Au lieu d’attribuer une probabilité de 1 à la classe correcte et 0 aux autres (étiquettes “one-hot”), on répartit légèrement la probabilité sur toutes les classes.  
Par exemple, la classe cible peut avoir 0,9 tandis que les autres reçoivent une petite valeur comme 0,1 répartie entre elles.  

Cette technique évite que le modèle devienne trop confiant, réduit le surapprentissage et améliore souvent les performances sur des données nouvelles.

### Dropout

**Dropout** est une technique de régularisation utilisée dans les réseaux de neurones pour réduire l’**overfitting**. L’idée est de **désactiver aléatoirement certains neurones pendant l’entraînement**.

À chaque itération, un neurone est mis à zéro avec une probabilité $(1 - p)$, où $(p)$ est la probabilité de le garder actif. Cela empêche le réseau de trop dépendre de certaines connexions et force le modèle à apprendre des **représentations plus robustes**.

Mathématiquement, si $(h_i)$ est la sortie d’un neurone, on applique un masque aléatoire $(m_i \sim Bernoulli(p))$ :

$$
\tilde{h}_i = m_i h_i
$$

Pendant l’inférence (test), le dropout est **désactivé** et toutes les unités du réseau sont utilisées.

En pratique, le dropout agit comme un **ensemble implicite de nombreux sous-réseaux**, ce qui améliore la généralisation du modèle.

In [7]:
# réduire le surapprentissage avec dropout et nc-dropout
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD none MODEL.LABEL_SMOOTHING 0.1 DROPOUT True

2026-03-07 12:48:16.819890: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 12:48:16.879027: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-07 12:48:18.406330: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'se_resnext50_32x4d'
100%|█| 49/49 [01:22<00:00,  1.68s/it, st

best val mae : 3.983


![Texte alternatif](Images/training_curves_none_label01_dropout.png)

In [8]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/last.pth MODEL.METHOD none MODEL.LABEL_SMOOTHING 0.1 DROPOUT True

2026-03-07 14:00:48.541083: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-07 14:00:48.613279: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-07 14:00:49.903163: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'se_resnext50_32x4d'
=> loading checkpoint './checkpoint/last.

test loss: 0.028
test mae: 4.937
test acc: 0.256
=> MAE by age group:
  0-17 yrs: mae=3.296, n=311, std(err)=4.858
  18-45 yrs: mae=4.363, n=1241, std(err)=4.094
  46-100 yrs: mae=7.805, n=426, std(err)=8.684


légèrement meilleure. on voit que le sur-apprentissage a réduit sur la courbe.

# 2. Méthodes d'estimation d'âge : DEX vs Residual

---

## 2.1 DEX Method

**DEX (Deep EXpectation)** est une approche de classification pour l’estimation de l’âge.

### Principe

- L’âge est considéré comme une **distribution de probabilité sur les classes d’âge** (0 à 100 ans).  
- Le modèle produit un vecteur de logits `cls_logits` de dimension `(batch_size, num_classes)` → chaque valeur correspond à la "probabilité" que l’âge appartienne à cette classe.  
- La prédiction finale de l’âge est calculée comme **l’espérance mathématique** de la distribution :

L'âge prédit $\hat{y}$ est calculé comme la somme pondérée de tous les âges possibles par leurs probabilités prédites :

$$
\hat{y} = \sum_{i=0}^{100} P(y=i) \cdot i
$$

où $P(y=i)$ est la probabilité que le modèle attribue à l'âge $i$.

### Loss

- La loss principale est une **CrossEntropy** :

La **loss** utilisée pour l'entraînement est la **cross-entropy** sur les logits de classification :

$$
\text{Loss} = \text{CrossEntropy}(cls\_logits)
$$

où `cls_logits` représente les sorties brutes du modèle avant application du softmax.


### Avantages

- Capture bien la **nature continue de l’âge** tout en utilisant une approche de classification.  
- L’espérance de la distribution permet de donner une estimation fine même si la classe maximale n’est pas exactement l’âge réel.




On entraine le modèle sur 60 epochs.

In [21]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex

I0000 00:00:1773581366.355229   58367 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'seresnext50_32x4d'
100%|█| 49/49 [01:20<00:00,  1.64s/it, stage=train, epoch=0, loss=0.0496, accN=0
100%|█| 12/12 [00:23<00:00,  1.98s/it, stage=val, epoch=0, loss=0.0289, acc=0.06
=> [epoch 000] best val mae was improved from 10000.000 to 7.443
100%|█| 49/49 [00:57<00:00,  1.17s/it, stage=train, epoch=1, loss=0.0457, accN=0
100%|█| 12/12 [00:07<00:00,  1.57it/s, stage=val, epoch=1, loss=0.0277, acc=0.07
=> [epoch 001] best val mae was improved from 7.443 to 6.677
100%|█| 49/49 [00:57<00:00,  1.18s/it, stage=train, epoch=2, loss=0.0437, accN=0
100%|█| 12/12 [00:07<00:00,  1.57it/s, stage=val, epoch=2, loss=0.0268, acc=0.06
=> [epoch 002] best val mae was improved from 6

Best val mae: 3.924


![Texte alternatif](Images/training_curves_dex02.png)

In [25]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch059_0.02349_3.9519.pth MODEL.METHOD dex 

I0000 00:00:1773585604.056345   82655 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'seresnext50_32x4d'
=> loading checkpoint './checkpoint/epoch059_0.02349_3.9519.pth'
=> loaded checkpoint './checkpoint/epoch059_0.02349_3.9519.pth'
=> start testing
100%|███████████████████████████████████████████| 25/25 [00:20<00:00,  1.23it/s]
test loss: 0.000
test mae: 4.762
test acc: 0.096
=> MAE by age group:
  0-17 yrs: mae=4.091, n=311, std(err)=5.800
  18-45 yrs: mae=3.817, n=1241, std(err)=3.219
  46-100 yrs: mae=8.004, n=426, std(err)=7.070


On peut observer que vers l'epoch 29, le MAE commence à se stabiliser. On a très peu de changement. 

### Dex Method with LabelSmoothing

In [26]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex MODEL.LABEL_SMOOTHING 0.05

I0000 00:00:1773585641.049157   82924 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'seresnext50_32x4d'
100%|█| 44/44 [01:32<00:00,  2.10s/it, stage=train, epoch=0, loss=0.0453, accN=0
100%|█| 19/19 [00:18<00:00,  1.01it/s, stage=val, epoch=0, loss=0.0477, acc=0.05
=> [epoch 000] best val mae was improved from 10000.000 to 7.513
100%|█| 44/44 [01:03<00:00,  1.43s/it, stage=train, epoch=1, loss=0.0418, accN=0
100%|█| 19/19 [00:07<00:00,  2.45it/s, stage=val, epoch=1, loss=0.0456, acc=0.06
=> [epoch 001] best val mae was improved from 7.513 to 7.243
100%|█| 44/44 [01:01<00:00,  1.41s/it, stage=train, epoch=2, loss=0.0402, accN=0
100%|█| 19/19 [00:07<00:00,  2.47it/s, stage=val, epoch=2, loss=0.0438, acc=0.09
=> [epoch 002] best val mae was improved from 7

best val mae : 4.276


![Texte alternatif](Images/training_curves_dex_labelsmoothing005.png)

In [30]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch027_0.03983_4.2687.pth MODEL.METHOD dex MODEL.LABEL_SMOOTHING 0.05

I0000 00:00:1773590038.666508  108013 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'seresnext50_32x4d'
=> loading checkpoint './checkpoint/epoch027_0.03983_4.2687.pth'
=> loaded checkpoint './checkpoint/epoch027_0.03983_4.2687.pth'
=> start testing
100%|███████████████████████████████████████████| 22/22 [00:22<00:00,  1.04s/it]
test loss: 0.000
test mae: 5.199
test acc: 0.101
=> MAE by age group:
  0-17 yrs: mae=5.259, n=311, std(err)=5.171
  18-45 yrs: mae=3.974, n=1241, std(err)=3.268
  46-100 yrs: mae=8.724, n=426, std(err)=8.057


In [31]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex MODEL.LABEL_SMOOTHING 0.1

I0000 00:00:1773590111.541379  108336 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'seresnext50_32x4d'
100%|█| 44/44 [01:29<00:00,  2.03s/it, stage=train, epoch=0, loss=0.0459, accN=0
100%|█| 17/17 [00:12<00:00,  1.37it/s, stage=val, epoch=0, loss=0.0457, acc=0.03
=> [epoch 000] best val mae was improved from 10000.000 to 9.012
100%|█| 44/44 [01:02<00:00,  1.42s/it, stage=train, epoch=1, loss=0.0428, accN=0
100%|█| 17/17 [00:07<00:00,  2.20it/s, stage=val, epoch=1, loss=0.0419, acc=0.06
=> [epoch 001] best val mae was improved from 9.012 to 6.653
100%|█| 44/44 [01:01<00:00,  1.40s/it, stage=train, epoch=2, loss=0.0413, accN=0
100%|█| 17/17 [00:07<00:00,  2.26it/s, stage=val, epoch=2, loss=0.0415, acc=0.08
=> [epoch 002] best val mae was improved from 6

![Texte alternatif](Images/training_curves_dexlabel01.png)

In [33]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch024_0.03780_4.6675.pth MODEL.METHOD dex MODEL.LABEL_SMOOTHING 0.1

I0000 00:00:1773594384.970653  132603 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'seresnext50_32x4d'
=> loading checkpoint './checkpoint/epoch024_0.03780_4.6675.pth'
=> loaded checkpoint './checkpoint/epoch024_0.03780_4.6675.pth'
=> start testing
100%|███████████████████████████████████████████| 22/22 [00:21<00:00,  1.01it/s]
test loss: 0.000
test mae: 5.345
test acc: 0.101
=> MAE by age group:
  0-17 yrs: mae=6.290, n=311, std(err)=4.527
  18-45 yrs: mae=3.910, n=1241, std(err)=3.181
  46-100 yrs: mae=8.834, n=426, std(err)=7.164


---


## 2.1 Residual Method

**La méthode “Residual” combine classification + régression du résidu pour améliorer la précision.**

### Principe

1. **Classification coarse** :  
   - Le modèle prédit `cls_logits` → la classe d’âge la plus probable (par exemple chaque classe représente 1 an).  
   - Cela donne une estimation approximative : `pred_class = cls_logits.argmax(dim=1)`

2. **Régression du résidu**  

- Le modèle prédit un **résidu** (`residual`) correspondant à la différence entre l’âge réel et la classe prédite :

$$
residual = age\_true - pred\_class
$$

- La **prédiction finale** est :

$$
\hat{y} = pred\_class + residual
$$

### Loss

- La **loss combinée** utilisée pour l’entraînement est :

$$
\text{Loss} = \text{CrossEntropy}(cls\_logits, age\_labels) + \alpha \cdot \text{L1Loss}(residual, residual\_target)
$$

- Ici, $\alpha$ contrôle le poids de la partie résiduelle dans la loss.

### Avantages

- Corrige la **précision fine** après la classification coarse.  
- Permet de capturer des écarts de quelques années que la classification seule ne peut pas résoudre.



In [6]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD residual

I0000 00:00:1773658066.083697   32701 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773658066.142900   32701 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773658067.641623   32701 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'seresnext50_32x4d'
model.safetensors: 100%|█████████████████████| 111M/111M [00:02<00:00, 49.6MB/s]
100%

![texte_altern](Images/training_curves_residual.png)

In [7]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch059_0.02328_3.8159.pth MODEL.METHOD residual

I0000 00:00:1773663594.104230   59373 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773663594.169031   59373 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773663595.519127   59373 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


=> creating model 'seresnext50_32x4d'
=> loading checkpoint './checkpoint/epoch059_0.02328_3.8159.pth'
=> loaded checkpoint './checkpoint/epoch059_0.02328_3.8159.pth'
=> start testing
100%|███████████████████████████████████████████| 16/16 [00:31<00:00,  1.98s/it]
test loss: 0.000
test mae: 4.730
test acc: 0.096
=> MAE by age group:
  0-17 yrs: mae=3.731, n=311, std(err)=5.853
  18-45 yrs: mae=3.935, n=1241, std(err)=3.404
  46-100 yrs: mae=7.775, n=426, std(err)=6.689


In [ ]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD residual DROPOUT True

I0000 00:00:1773767739.966681   33151 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'seresnext50_32x4d'
100%|█| 44/44 [01:58<00:00,  2.70s/it, stage=train, epoch=0, loss=0.0456, accN=0
100%|█| 12/12 [00:31<00:00,  2.60s/it, stage=val, epoch=0, loss=0.0303, acc=0.04
=> [epoch 000] best val mae was improved from 10000.000 to 8.973
100%|█| 44/44 [01:15<00:00,  1.71s/it, stage=train, epoch=1, loss=0.0413, accN=0
100%|█| 12/12 [00:13<00:00,  1.16s/it, stage=val, epoch=1, loss=0.0277, acc=0.07
=> [epoch 001] best val mae was improved from 8.973 to 6.413
100%|█| 44/44 [01:19<00:00,  1.80s/it, stage=train, epoch=2, loss=0.0394, accN=0
100%|█| 12/12 [00:13<00:00,  1.13s/it, stage=val, epoch=2, loss=0.028, acc=0.048
=> [epoch 002] best val mae was not improved fr


![residualdrop](Images/training_curves_residual_dropout.png)

In [38]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/best.pth MODEL.METHOD residual

I0000 00:00:1773498712.053104  243370 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773498712.121621  243370 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773498713.409878  243370 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'seresnext50_32x4d'
=> loading checkpoint './checkpoint/best.pth'
=> loaded checkpoint './checkpoint/best

In [40]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD residual TRAIN.LR 3e-4

I0000 00:00:1773498980.419773  244824 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773498980.476856  244824 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773498981.805671  244824 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'seresnext50_32x4d'
100%|█| 49/49 [01:23<00:00,  1.70s/it, stage=train, epoch=0, loss=0.0519, accN=0
100%

In [41]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/last.pth MODEL.METHOD residual TRAIN.LR 3e-4

I0000 00:00:1773503460.141081  269345 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773503460.205873  269345 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773503461.537683  269345 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'seresnext50_32x4d'
=> loading checkpoint './checkpoint/last.pth'
=> loaded checkpoint './checkpoint/last


### Résumé comparatif

| Méthode   | Type                          | Prédiction finale          | Points forts                          |
|-----------|-------------------------------|---------------------------|--------------------------------------|
| DEX       | Classification               | Espérance des classes     | Stable, capture distribution continue|
| Residual  | Classification + Régression  | Classe + résidu           | Précision fine, corrige la classe prédite|

---


## Modélisation de l’incertitude

Au lieu de prédire uniquement un âge, le réseau prédit :

- **μ** : âge estimé  
- **σ²** ou **b** : incertitude

### 1. Perte Laplacienne

On suppose que l’âge suit une **distribution de Laplace** :

$$
p(y|x) = Laplace(\mu(x), b(x))
$$

Loss :

$$
L = \frac{|y-\mu|}{b} + \log(b)
$$

- Robuste aux outliers  
- Erreur absolue au lieu de quadratique  
- Adapté aux labels subjectifs

In [ ]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD laplace

I0000 00:00:1773595179.143184  135290 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'seresnext50_32x4d'
100%|█| 44/44 [01:24<00:00,  1.93s/it, stage=train, epoch=0, loss=0.0583, accN=0
100%|█| 17/17 [00:11<00:00,  1.45it/s, stage=val, epoch=0, loss=0.0463, acc=0.01
=> [epoch 000] best val mae was improved from 10000.000 to 23.704
100%|█| 44/44 [00:58<00:00,  1.34s/it, stage=train, epoch=1, loss=0.0417, accN=0
100%|█| 17/17 [00:07<00:00,  2.28it/s, stage=val, epoch=1, loss=0.0362, acc=0.04
=> [epoch 001] best val mae was improved from 23.704 to 10.250
100%|█| 44/44 [00:59<00:00,  1.34s/it, stage=train, epoch=2, loss=0.0335, accN=0
100%|█| 17/17 [00:07<00:00,  2.29it/s, stage=val, epoch=2, loss=0.0329, acc=0.06
=> [epoch 002] best val mae was improved fro

best val mae: 4.001

Le model avec laPlaceLoss au lieu de crossEntropyLoss converge beaucoup plus vite.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tf_log MODEL.METHOD laplace

![Texte alternatif](Images/laplace.png)

In [2]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch044_0.02729_4.0007.pth MODEL.METHOD laplace

I0000 00:00:1773602035.107538  169618 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'seresnext50_32x4d'
=> loading checkpoint './checkpoint/epoch044_0.02729_4.0007.pth'
=> loaded checkpoint './checkpoint/epoch044_0.02729_4.0007.pth'
=> start testing
100%|███████████████████████████████████████████| 22/22 [00:22<00:00,  1.03s/it]
test loss: 0.000
test mae: 4.600
test acc: 0.105
=> MAE by age group:
  0-17 yrs: mae=3.072, n=311, std(err)=4.181
  18-45 yrs: mae=3.960, n=1241, std(err)=3.409
  46-100 yrs: mae=7.582, n=426, std(err)=7.531


### 2. Perte Gaussienne

On suppose que l’âge réel suit une **distribution normale** :

$$
p(y|x) = \mathcal{N}(\mu(x), \sigma^2(x))
$$

Loss :

$$
L = \frac{(y-\mu)^2}{2\sigma^2} + \frac{1}{2}\log(\sigma^2)
$$

- Sensible aux outliers  
- Variance adaptative selon la difficulté de l’image  
- Robuste aux labels bruités

In [ ]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD gaussian

best val mae: 4.359

![Texte alternatif](Images/gaussianTrain.png)

![Texte alternatif](Images/gaussian.png)

In [ ]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch046_0.06742_4.2653.pth MODEL.METHOD gaussian

![Texte alternatif](Images/gaussianTest.png)

# Methode Ordinal

In [23]:
!python train_classification.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD ordinal

I0000 00:00:1773758795.387884   20540 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating classification model 'se_resnext50_32x4d' (method=ordinal)
=> [epoch 000] best val mae was improved from 10000.000 to 7.583
=> [epoch 001] best val mae was improved from 7.583 to 6.493
=> [epoch 002] best val mae was improved from 6.493 to 6.222
=> [epoch 003] best val mae was improved from 6.222 to 5.071
=> [epoch 004] best val mae was not improved from 5.071 (5.893)
=> [epoch 005] best val mae was not improved from 5.071 (5.672)
=> [epoch 006] best val mae was not improved from 5.071 (5.233)
=> [epoch 007] best val mae was not improved from 5.071 (6.353)
=> [epoch 008] best val mae was improved from 5.071 to 5.000
=> [epoch 009] best val mae was not improved from 5.000 (5.39

![Texte alternatif](Images/ordinal.png)

In [21]:
!python test_classification.py --data_dir ./appa-real-release --resume ./checkpoint_cls/epoch055_0.00073_3.8321.pth MODEL.METHOD ordinal

I0000 00:00:1773758451.901450   19942 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating classification model 'se_resnext50_32x4d' (method=ordinal)
=> loading classification checkpoint './checkpoint_cls/epoch055_0.00073_3.8321.pth'
=> loaded checkpoint './checkpoint_cls/epoch055_0.00073_3.8321.pth' (epoch 56)
=> start classification testing
test accuracy (cls): 0.0000
test mae (cls, global): 4.464
=> MAE by age group (classification):
  0-17 yrs: mae=3.662, n=311, std(err)=5.335
  18-45 yrs: mae=3.787, n=1241, std(err)=3.147
  46-100 yrs: mae=7.023, n=426, std(err)=6.669


## 2.5 Analyse globale

mettre image avec les courbes de validation superposées et dans un graphique à part à coté superposée également.

![image.png](Images/globaleloss.png)
![image.png](Images/globaleMAE.png)
![image.png](Images/dexglobalelabelsmooth.png)
![image.png](Images/lossdexlabelsmooth.png)

# 3. Traitements d'images

Nous avons remplacé dataset.py qui utilise imgaug par un package plus récent : albumentations. La version récente se trouve dans le fichier **dataset_recent.py**.

In [8]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD none

2026-03-02 11:25:22.129630: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 11:25:22.185182: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 11:25:23.619607: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'se_resnext50_32x4d'
100%|█| 49/49 [01:21<00:00,  1.67s/it, st

additional opts: ['MODEL.METHOD', 'none']
best val mae: 4.122

![Texte alternatif](Images/training_curves_DatasetRecent.png)

In [9]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch058_0.03712_4.1220_0.11400.pth MODEL.METHOD none

2026-03-02 12:38:36.014905: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 12:38:36.080847: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 12:38:37.473824: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'se_resnext50_32x4d'
=> loading checkpoint './checkpoint/epoch

## 3.2 Test-Time-Augmentation (TTA)



La **Test-Time Augmentation (TTA)** est une technique utilisée lors de l’inférence pour améliorer la robustesse des prédictions d’un modèle. Au lieu de faire une prédiction à partir d’une seule image, on applique **plusieurs transformations simples** à l’image (par exemple : flip horizontal, rotation ou changement d’échelle). Le modèle effectue ensuite une prédiction pour chaque version transformée.

Les différentes prédictions sont ensuite **agrégées**, généralement en calculant leur **moyenne**, afin d’obtenir une estimation finale plus stable. On peut également calculer **l’écart-type des prédictions**, qui donne une indication sur l’**incertitude du modèle**.

Dans notre cas, nous testons deux approches :
- une version **aléatoire**, qui applique des transformations différentes à chaque passage ;
- une version **déterministe**, qui utilise un ensemble fixe de transformations.

L’inconvénient principal de cette méthode est l’augmentation du **temps d’inférence**. En effet, si l’on applique \(k\) augmentations, le modèle doit effectuer environ **\(k\) passes forward**, ce qui multiplie le temps de prédiction par ce facteur.

In [ ]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex TTA 1

In [7]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch048_0.02341_3.9303.pth MODEL.METHOD dex TTA 1

I0000 00:00:1773701294.989874   67451 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773701295.063366   67451 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773701296.587052   67451 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'seresnext50_32x4d'
=> loading checkpoint './checkpoint/epoch048_0.02341_3.9303.pth'
=> loaded checkpoint

In [3]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex TTA 2

I0000 00:00:1773687444.123684   10390 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773687444.184473   10390 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773687448.899659   10390 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'seresnext50_32x4d'
model.safetensors: 100%|█████████████████████| 111M/111M [00:02<00:00, 49.7MB/s]
100%

![image.png](Images/training_curves_Dex_tta1.png)

In [5]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch023_0.02340_3.8526.pth MODEL.METHOD dex TTA 2

I0000 00:00:1773694598.427895   38926 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773694598.484003   38926 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773694599.642110   38926 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'seresnext50_32x4d'
=> loading checkpoint './checkpoint/epoch023_0.02340_3.8526.pth'
=> loaded checkpoint

![image.png](Images/training_curves_dex_tta2.png)

# 4. Changements de backbone: resNet vs EfficientNet


Dans cette section, nous explorons l’impact du choix du **backbone** sur la performance de nos modèles pour la prédiction de l’âge apparent. Le backbone correspond à la partie du réseau responsable de l’extraction des **features visuelles** à partir des images, et joue un rôle clé dans la qualité des représentations apprises.  

Nous comparons deux architectures populaires :  

1. **ResNet (Residual Network)** :  
   - Introduit les **blocs résiduels**, qui facilitent l’apprentissage dans les réseaux très profonds.  
   - Très robuste et éprouvé sur de nombreux benchmarks, mais peut être moins efficace paramétriquement pour des modèles de taille moyenne.  

2. **EfficientNet** :  
   - Réseau récent optimisé pour un bon compromis **performance / taille / rapidité**.  
   - Utilise une **mise à l’échelle uniforme** des dimensions profondeur, largeur et résolution, ce qui permet d’obtenir de meilleures performances avec moins de paramètres.  

L’objectif de cette comparaison est d’analyser comment le choix du backbone influence :  
- la précision de prédiction des modèles DEX et Residual DEX,  
- la qualité des features extraites pour la régression de l’âge,  
- et le compromis entre **performance et complexité computationnelle**.

In [ ]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD laplace MODEL.ARCH "seresnet50"

2026-03-01 18:30:41.702353: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-01 18:30:41.766255: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-01 18:30:43.215162: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'se_resnet50'
Downloading: "http://data.lip6.fr/cadene/pretrai

moins précis que se_resnext50_32x4d.

In [17]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD laplace MODEL.ARCH "resnet101"

2026-03-01 19:45:32.246190: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-01 19:45:32.347863: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-01 19:45:33.830819: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'resnet101'
/opt/python/lib/python3.13/site-packages/torchvisi

![Texte alternatif](Images/training_curves_laplace_resnet101.png)

Converge très lentement par rapport à resnet50. 

Pour utiliser le backbone efficientnet_b0, il a fallu utiliser le package timm car il n'est pas contenu dans pretrained_models. 

In [16]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD laplace MODEL.ARCH "efficientnet_b0"

2026-03-02 14:14:39.062895: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 14:14:39.121020: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 14:14:40.609886: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'efficientnet_b0'
100%|█| 49/49 [00:40<00:00,  1.20it/s, stage

best val mae : 4.180

![Texte alternatif](Images/training_curves_laplaceefficientb0.png)

très rapide (31 min contre 1h généralement). 

In [ ]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch054_0.03037_4.1801_0.00000.pth MODEL.METHOD laplace MODEL.ARCH "efficientnet_b0" 

test loss: 0.032
test mae: 4.810
=> MAE by age group:
  0-17 yrs: mae=3.184, n=311, std(err)=4.775
  18-45 yrs: mae=4.258, n=1241, std(err)=3.882
  46-100 yrs: mae=7.601, n=426, std(err)=7.701

Plus moderne et rapide. mais sur apprentissage visible. Testons dropout pour réduire le sur-apprentissage.


In [24]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD laplace MODEL.ARCH "efficientnet_b0" DROPOUT True

2026-03-02 16:08:24.270245: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 16:08:24.325990: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 16:08:25.697365: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'efficientnet_b0'
100%|█| 49/49 [00:40<00:00,  1.20it/s, stage

best val mae : 4.306

![Texte alternatif](Images/training_curves_efficientb0_laplace_dropout.png)

In [28]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch050_0.03081_4.3061_0.00000.pth MODEL.METHOD laplace MODEL.ARCH "efficientnet_b0" DROPOUT True

2026-03-02 16:42:46.666527: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 16:42:46.725428: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 16:42:48.056744: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'efficientnet_b0'
=> loading checkpoint './checkpoint/epoch050

se généralise mal.

In [12]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD laplace MODEL.ARCH "efficientnet_b3"

I0000 00:00:1773616294.433547  248948 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'efficientnet_b3'
100%|█| 47/47 [00:42<00:00,  1.10it/s, stage=train, epoch=0, loss=0.0643, accN=0
100%|█| 17/17 [00:06<00:00,  2.58it/s, stage=val, epoch=0, loss=0.0472, acc=0.01
=> [epoch 000] best val mae was improved from 10000.000 to 26.159
100%|█| 47/47 [00:38<00:00,  1.22it/s, stage=train, epoch=1, loss=0.0473, accN=0
100%|█| 17/17 [00:04<00:00,  3.42it/s, stage=val, epoch=1, loss=0.0432, acc=0.00
=> [epoch 001] best val mae was improved from 26.159 to 18.371
100%|█| 47/47 [00:38<00:00,  1.21it/s, stage=train, epoch=2, loss=0.0400, accN=0
100%|█| 17/17 [00:04<00:00,  3.52it/s, stage=val, epoch=2, loss=0.0337, acc=0.06
=> [epoch 002] best val mae was improved from 

![Texte alternatif](Images/laplaceefficientb3.png)

In [ ]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch059_0.02652_3.8976.pth MODEL.METHOD laplace MODEL.ARCH "efficientnet_b3" 

test loss: 0.032
test mae: 4.799

=> MAE by age group:
  0-17 yrs: mae=3.243, n=311, std(err)=4.747
  18-45 yrs: mae=4.046, n=1241, std(err)=3.498
  46-100 yrs: mae=8.130, n=426, std(err)=8.110

In [ ]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex MODEL.ARCH "efficientnet_b0"

In [50]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/best.pth MODEL.METHOD dex MODEL.ARCH "efficientnet_b0" 

I0000 00:00:1773520887.487455  366493 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773520887.550768  366493 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773520888.980231  366493 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'efficientnet_b0'
=> loading checkpoint './checkpoint/best.pth'
=> loaded checkpoint './checkpoint/best.p

test mae: 4.969
test acc: 0.086
=> MAE by age group:
  0-17 yrs: mae=4.336, n=311, std(err)=7.022
  18-45 yrs: mae=4.032, n=1241, std(err)=3.904
  46-100 yrs: mae=8.161, n=426, std(err)=7.085


In [ ]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex MODEL.ARCH "seresnet50"

I0000 00:00:1773478872.964659  124226 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773478873.028707  124226 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773478874.444047  124226 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'seresnet50'
100%|█| 49/49 [01:00<00:00,  1.24s/it, stage=train, epoch=0, loss=0.0512, accN=0
100%|█| 12/

In [28]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/best.pth MODEL.METHOD dex MODEL.ARCH "seresnet50" 

I0000 00:00:1773482334.807910  146839 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773482334.878404  146839 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773482336.287396  146839 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'seresnet50'
=> loading checkpoint './checkpoint/best.pth'
=> loaded checkpoint './checkpoint/best.pth'
=

In [29]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex MODEL.ARCH "resnet101"

I0000 00:00:1773482504.744359  147609 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773482504.799530  147609 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773482506.332467  147609 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'resnet101'
model.safetensors: 100%|█████████████████████| 179M/179M [00:11<00:00, 15.4MB/s]
100%|█| 49/4

In [30]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/best.pth MODEL.METHOD dex MODEL.ARCH "resnet101" 

I0000 00:00:1773487090.610754  172150 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773487090.678464  172150 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773487092.005067  172150 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'resnet101'
=> loading checkpoint './checkpoint/best.pth'
=> loaded checkpoint './checkpoint/best.pth'
=>

In [32]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex MODEL.ARCH "efficientnet_b3"

I0000 00:00:1773487217.936491  172741 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773487217.995537  172741 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773487219.422878  172741 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'efficientnet_b3'
model.safetensors: 100%|███████████████████| 49.3M/49.3M [00:01<00:00, 34.5MB/s]
100%|█

In [33]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/best.pth MODEL.METHOD dex MODEL.ARCH "efficientnet_b3" 

I0000 00:00:1773491227.412191  196083 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773491227.472968  196083 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773491228.625425  196083 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'efficientnet_b3'
=> loading checkpoint './checkpoint/best.pth'
=> loaded checkpoint './checkpoint/best.p

### Tableau récapitulatif

---

# 5. Plus grand pas 

On choisit de prendre le modèle qui a donnée le meilleure mae.

In [40]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex TRAIN.LR 0.1 TRAIN.LR_DECAY_STEP 5 

2026-03-02 21:16:49.874143: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 21:16:49.928277: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 21:16:51.284062: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'se_resnext50_32x4d'
100%|█| 49/49 [01:21<00:00,  1.67s/it, st

![Texte alternatif](Images/training_curves_lr01_decaystep5.png)

In [41]:

!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch009_0.05095_11.3632_0.00000.pth MODEL.METHOD dex TRAIN.LR 0.1 TRAIN.LR_DECAY_STEP 5 

2026-03-02 22:33:27.510214: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 22:33:27.569190: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 22:33:28.830327: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'se_resnext50_32x4d'
=> loading checkpoint './checkpoint/epoch

In [34]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex TRAIN.LR 3e-4 MODEL.ARCH "efficientnet_b0" 

I0000 00:00:1773491377.967349  196814 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773491378.031594  196814 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773491379.381133  196814 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'efficientnet_b0'
100%|█| 49/49 [00:40<00:00,  1.20it/s, stage=train, epoch=0, loss=0.0520, accN=0
100%|█

In [36]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/best.pth MODEL.METHOD dex TRAIN.LR 3e-4 MODEL.ARCH "efficientnet_b0" 

I0000 00:00:1773494131.775327  218453 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773494131.841469  218453 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773494133.100656  218453 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'efficientnet_b0'
=> loading checkpoint './checkpoint/best.pth'
=> loaded checkpoint './checkpoint/best.p

In [17]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex TRAIN.LR 3e-4 MODEL.ARCH "efficientnet_b3" 

I0000 00:00:1773620459.925156  283210 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'efficientnet_b3'
100%|█| 47/47 [00:44<00:00,  1.06it/s, stage=train, epoch=0, loss=0.0488, accN=0
100%|█| 17/17 [00:07<00:00,  2.38it/s, stage=val, epoch=0, loss=0.0428, acc=0.03
=> [epoch 000] best val mae was improved from 10000.000 to 9.643
100%|█| 47/47 [00:40<00:00,  1.15it/s, stage=train, epoch=1, loss=0.0437, accN=0
100%|█| 17/17 [00:05<00:00,  3.28it/s, stage=val, epoch=1, loss=0.039, acc=0.058
=> [epoch 001] best val mae was improved from 9.643 to 6.450
100%|█| 47/47 [00:41<00:00,  1.14it/s, stage=train, epoch=2, loss=0.0410, accN=0
100%|█| 17/17 [00:05<00:00,  3.37it/s, stage=val, epoch=2, loss=0.0381, acc=0.05
=> [epoch 002] best val mae was improved from 6.4

In [18]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch037_0.03364_4.1757.pth MODEL.METHOD dex TRAIN.LR 3e-4 MODEL.ARCH "efficientnet_b3" 

I0000 00:00:1773626066.160354  308920 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'efficientnet_b3'
=> loading checkpoint './checkpoint/epoch037_0.03364_4.1757.pth'
=> loaded checkpoint './checkpoint/epoch037_0.03364_4.1757.pth'
=> start testing
100%|███████████████████████████████████████████| 22/22 [00:08<00:00,  2.66it/s]
test loss: 0.000
test mae: 4.814
test acc: 0.096
=> MAE by age group:
  0-17 yrs: mae=3.913, n=311, std(err)=5.198
  18-45 yrs: mae=3.951, n=1241, std(err)=3.450
  46-100 yrs: mae=7.985, n=426, std(err)=7.058


In [4]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex TRAIN.LR 1e-4 MODEL.ARCH "efficientnet_b3" 

I0000 00:00:1773653849.300109    8224 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773653849.368783    8224 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773653851.007650    8224 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'efficientnet_b3'
100%|█| 47/47 [01:15<00:00,  1.61s/it, stage=train, epoch=0, loss=0.0516, accN=0
100%|█

In [5]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/epoch056_0.02405_4.3770.pth MODEL.METHOD dex TRAIN.LR 1e-4 MODEL.ARCH "efficientnet_b3" 

I0000 00:00:1773657962.104900   31891 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773657962.165219   31891 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773657963.482468   31891 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'efficientnet_b3'
=> loading checkpoint './checkpoint/epoch056_0.02405_4.3770.pth'
=> loaded checkpoint '

![image.png](Images/globalelr.png)

## Analyse de la distribution des âges dans le dataset

Lors de nos expérimentations, nous avons observé que le modèle prédit moins précisément les âges situés dans les intervalles **0–20 ans** et **80–100 ans**. Une hypothèse naturelle pour expliquer cette baisse de performance est liée à la **répartition déséquilibrée des exemples dans le dataset**.

### Distribution dans l'ensemble d'entraînement

![Distribution des âges dans l'ensemble d'entraînement](Images/age_distribution_gt_avg_train.png)

La figure ci-dessus montre la distribution des âges dans l’ensemble **d’entraînement**. On observe plusieurs caractéristiques importantes :

- La distribution est **fortement concentrée entre 20 et 40 ans**, avec un **pic autour de 25 ans**.
- Le nombre d'exemples diminue progressivement après **40 ans**.
- Les âges supérieurs à **60 ans** sont **très peu représentés**.
- Les âges extrêmes (**0–10 ans** et **80–100 ans**) contiennent **très peu d’exemples**.

Cette distribution indique que le modèle est principalement entraîné sur des visages appartenant à des **adultes jeunes**, ce qui peut biaiser l’apprentissage. Le réseau apprend donc mieux les caractéristiques associées à ces âges majoritaires, tandis que les âges rares disposent de **moins d'exemples pour apprendre des représentations robustes**.

### Distribution dans les ensembles de validation et de test

Les distributions des ensembles de **validation** et de **test** présentent un comportement similaire :

![Distribution des âges dans l'ensemble de validation](Images/age_distribution_gt_avg_valid.png)

![Distribution des âges dans l'ensemble de test](Images/age_distribution_gt_avg_test.png)

On retrouve la même **asymétrie dans la distribution des âges**, avec une forte concentration entre **20 et 40 ans** et une sous-représentation des âges très jeunes et très élevés.

### Impact sur les performances du modèle

Ce déséquilibre peut expliquer pourquoi le modèle obtient de **moins bonnes performances sur les âges rares**, notamment :

- **0–20 ans**
- **80–100 ans**

En effet, les modèles d'apprentissage profond sont sensibles à la distribution des données : les classes majoritaires influencent davantage la fonction de perte pendant l'entraînement. Le modèle tend donc à **optimiser ses prédictions pour les classes les plus fréquentes**, au détriment des classes rares.

### Pistes d'amélioration

Pour atténuer cet effet, plusieurs approches peuvent être envisagées :

- utiliser une **fonction de perte pondérée (weighted loss)** pour donner plus d’importance aux âges rares ;
- appliquer des techniques d’**augmentation de données ciblées** sur les classes sous-représentées ;
- utiliser des méthodes de **rééchantillonnage (oversampling)** pour équilibrer les classes d’âge pendant l’entraînement.

Ces approches permettraient d'améliorer la capacité du modèle à **généraliser sur l’ensemble du spectre des âges**.

In [47]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD weightLoss

I0000 00:00:1773513639.136391  320558 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773513639.206926  320558 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773513640.587726  320558 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'seresnext50_32x4d'
100%|█| 49/49 [01:22<00:00,  1.69s/it, stage=train, epoch=0, loss=0.0521, accN=0
100%

![Distribution des âges train](Images/age_distribution_gt_avg_train.png)


In [48]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/last.pth MODEL.METHOD weightLoss

I0000 00:00:1773518348.145298  344849 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773518348.207963  344849 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773518349.513298  344849 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
=> creating model 'seresnext50_32x4d'
=> loading checkpoint './checkpoint/last.pth'
=> loaded checkpoint './checkpoint/last

In [3]:

!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD dex MODEL.balanced_sampler True

Traceback (most recent call last):
  File "/home/barkan/Cours 2025-2026/deepLearning/Project-age-estimation-pytorch/train.py", line 5, in <module>
    import better_exceptions
ModuleNotFoundError: No module named 'better_exceptions'


=> MAE by age group:
  0-17 yrs: mae=3.232, n=311, std(err)=5.117
  18-45 yrs: mae=4.155, n=1241, std(err)=3.585
  46-100 yrs: mae=7.583, n=426, std(err)=7.556

In [24]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/last.pth MODEL.METHOD dex MODEL.balanced_sampler True

I0000 00:00:1773073078.851462   55403 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'se_resnext50_32x4d'
=> loading checkpoint './checkpoint/last.pth'
=> loaded checkpoint './checkpoint/last.pth'
=> start testing
100%|█| 16/16 [00:23<00:00,  1.46s/it, stage=val, epoch=0, loss=0.0251, mae=4.81
test loss: 0.025
test mae: 4.811
test acc: 0.190
=> MAE by age group:
  0-17 yrs: mae=3.703, n=311, std(err)=6.189
  18-45 yrs: mae=4.331, n=1241, std(err)=3.979
  46-100 yrs: mae=7.018, n=426, std(err)=6.278


contre : => MAE by age group:
  0-17 yrs: mae=3.232, n=311, std(err)=5.117
  18-45 yrs: mae=4.155, n=1241, std(err)=3.585
  46-100 yrs: mae=7.583, n=426, std(err)=7.556

In [10]:
!python train.py --data_dir ./appa-real-release --tensorboard tf_log MODEL.METHOD balancedSoftmax

Traceback (most recent call last):
  File "/home/barkan/Cours 2025-2026/deepLearning/Project-age-estimation-pytorch/train.py", line 5, in <module>
    import better_exceptions
ModuleNotFoundError: No module named 'better_exceptions'


In [29]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/last.pth MODEL.METHOD balancedSoftmax

I0000 00:00:1773079779.068774   78898 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'se_resnext50_32x4d'
=> loading checkpoint './checkpoint/last.pth'
=> loaded checkpoint './checkpoint/last.pth'
=> start testing
100%|█| 16/16 [00:22<00:00,  1.42s/it, stage=val, epoch=0, loss=0.0251, mae=5.26
test loss: 0.025
test mae: 5.269
test acc: 0.169
=> MAE by age group:
  0-17 yrs: mae=3.541, n=311, std(err)=7.049
  18-45 yrs: mae=5.296, n=1241, std(err)=4.874
  46-100 yrs: mae=6.453, n=426, std(err)=7.360


In [30]:
!python test.py --data_dir ./appa-real-release --resume ./checkpoint/last.pth MODEL.METHOD balancedSoftmax MC_DROPOUT True

I0000 00:00:1773079828.612149   79073 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'se_resnext50_32x4d'
=> loading checkpoint './checkpoint/last.pth'
=> loaded checkpoint './checkpoint/last.pth'
=> start testing
100%|█| 16/16 [05:07<00:00, 19.22s/it, stage=val, epoch=0, loss=0.0251, mae=5.26
Figure(640x480)
Figure(640x480)
/opt/python/lib/python3.13/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/python/lib/python3.13/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
Correlation between uncertainty and error: nan
test loss: 0.025
test mae: 5.269
test acc: 0.169
=> MAE by age group:
  0-17 yrs: mae=3.541,

### Utilisation potentielle des GAN pour équilibrer la distribution des âges

Une autre approche possible pour atténuer le déséquilibre des données consiste à utiliser des **GAN (Generative Adversarial Networks)** afin de générer artificiellement de nouvelles images pour les classes d’âge sous-représentées.

Les **GAN** sont des modèles génératifs composés de deux réseaux de neurones entraînés simultanément :

- **Le générateur (Generator)** : il produit des images synthétiques à partir d’un bruit aléatoire.
- **Le discriminateur (Discriminator)** : il tente de distinguer les images réelles du dataset des images générées.

Pendant l'entraînement, ces deux réseaux s'affrontent dans un jeu adversarial : le générateur cherche à produire des images de plus en plus réalistes afin de tromper le discriminateur, tandis que le discriminateur apprend à mieux détecter les images synthétiques. À l'équilibre, le générateur peut produire des images très proches des données réelles.

Dans le contexte de l’estimation d’âge, les GAN pourraient être utilisés pour **générer de nouveaux visages appartenant aux classes d’âge rares**, par exemple :

- **0–10 ans**
- **80–100 ans**

Ces images synthétiques pourraient ensuite être ajoutées au dataset d'entraînement afin de **rééquilibrer la distribution des âges**. Cette stratégie permettrait au modèle de voir davantage d'exemples pour ces classes, ce qui pourrait améliorer ses performances sur les âges extrêmes.

Cependant, l'utilisation de GAN présente également certaines limites :

- l'entraînement des GAN est **complexe et parfois instable** ;
- les images générées doivent être **suffisamment réalistes** pour ne pas introduire de bruit dans l’apprentissage ;
- il est nécessaire de vérifier que les images synthétiques **représentent correctement l’âge ciblé**.

Malgré ces difficultés, les GAN restent une approche prometteuse pour **augmenter artificiellement les données dans les classes rares** et améliorer la robustesse des modèles d’estimation d’âge.


![age_distribution_UTKFace](age_distribution_fusion.png)

# Changement de jeu de donnée

## 1. UTKFace

Le dataset **UTKFace** est une base d'images faciales annotées par âge, sexe et ethnie, contenant plus de 20 000 photos de visages variés. Il est largement utilisé pour l'entraînement et l'évaluation de modèles de reconnaissance d'âge ou d'analyse démographique en vision par ordinateur.

In [ ]:

!pip install gdown
!gdown https://drive.google.com/uc?id=1mb5Z24TsnKI3ygNIlX6ZFiwUj0_PmpAW -O part1.tar.gz
!gdown https://drive.google.com/uc?id=19vdaXVRtkP-nyxz1MYwXiFsh_m_OL72b -O part2.tar.gz
!gdown https://drive.google.com/uc?id=1oj9ZWsLV2-k2idoW_nRSrLQLUP3hus3b -O part3.tar.gz


In [5]:
!python utkFace_dataset.py 

Extraction de part1.tar.gz ...
/home/onyxia/Project-age-estimation-pytorch/utkFace_dataset.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)
Extraction de part2.tar.gz ...
Extraction de part3.tar.gz ...
Total images trouvées: 24106
Skip 53__0_20170116184028385.jpg: invalid literal for int() with base 10: ''
Skip 39_1_20170116174525125.jpg: not enough values to unpack (expected 4, got 3)
Skip 61_3_20170109150557335.jpg: not enough values to unpack (expected 4, got 3)
Skip 61_1_20170109142408075.jpg: not enough values to unpack (expected 4, got 3)
Total images valides: 24102
✅ Extraction et préparation terminées !
Train: 16871, Valid: 3615, Test: 3616


Le lancement de cette commande dure environ 5h10.

In [7]:
!python train.py --data_dir ./UTKFace --tensorboard tf_log MODEL.METHOD dex

I0000 00:00:1773091789.466113   10601 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'se_resnext50_32x4d'
 64%|▋| 135/210 [03:32<01:43,  1.38s/it, stage=train, epoch=0, loss=0.0451, mae=libpng warning: iCCP: known incorrect sRGB profile
100%|█| 210/210 [05:16<00:00,  1.51s/it, stage=train, epoch=0, loss=0.0440, mae=
  0%|                                                    | 0/29 [00:00<?, ?it/s]Corrupt JPEG data: bad Huffman code
 38%|▍| 11/29 [00:20<00:13,  1.36it/s, stage=val, epoch=0, loss=0.0254, mae=8.32Corrupt JPEG data: premature end of data segment
100%|█| 29/29 [00:33<00:00,  1.16s/it, stage=val, epoch=0, loss=0.0261, mae=8.25
=> [epoch 000] best val mae was improved from 10000.000 to 8.256
 20%|▏| 41/210 [00:55<03:48,  1.35s/it, stage=train, ep

À partir de l'epoch 10‑11, le MAE sur l'ensemble de validation atteint une valeur proche de 5.32 et stagne ensuite autour de 5.2‑5.5 malgré la poursuite de l'entraînement. On observe que la loss d'entraînement continue à diminuer très légèrement, mais de manière quasi insignifiante, ce qui suggère que le réseau n'apprend presque plus. Ce comportement est typique d'un plateau d'apprentissage et peut indiquer une disparition partielle du gradient (vanishing gradient), phénomène fréquent dans les réseaux profonds comme SE-ResNeXt. Dans ce cas, les poids évoluent trop faiblement pour permettre une amélioration significative de la performance sur l'ensemble de validation.



![image_utkface_dex](Images/training_curves_Dex_UTKface.png)

On observe que la **loss sur l'ensemble de validation augmente rapidement**, tandis que la **loss sur l'ensemble d'entraînement continue de diminuer**. Ce comportement est caractéristique d’un **surapprentissage (overfitting)** : le modèle s’adapte de plus en plus aux données d'entraînement en apprenant certains motifs par cœur, mais sa capacité de généralisation sur de nouvelles données se dégrade.

Concernant le **MAE**, on constate également que la **courbe de validation stagne autour de 5**, alors que le **MAE sur l'ensemble d'entraînement continue de diminuer**. Cela confirme que les améliorations du modèle profitent principalement aux données d'entraînement sans améliorer les performances sur l'ensemble de validation.

In [9]:
!python test.py --data_dir ./UTKFace --resume ./checkpoint/last.pth MODEL.METHOD dex

I0000 00:00:1773109748.777620   50936 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
=> creating model 'se_resnext50_32x4d'
=> loading checkpoint './checkpoint/last.pth'
=> loaded checkpoint './checkpoint/last.pth'
=> start testing
100%|█| 29/29 [00:30<00:00,  1.05s/it, stage=val, epoch=0, loss=0.0527, mae=5.75
test loss: 0.053
test mae: 5.755
test acc: 0.221
=> MAE by age group:
  0-17 yrs: mae=3.568, n=704, std(err)=6.999
  18-45 yrs: mae=4.991, n=2078, std(err)=4.968
  46-100 yrs: mae=9.507, n=834, std(err)=9.711


### Distribution des ages
![age_distribution_UTKFace](Images/UTKFace_age_distribution_gt_avg_train.png)
![age_distribution_UTKFace](Images/UTKFace_age_distribution_gt_avg_test.png)
![age_distribution_UTKFace](Images/UTKFace_age_distribution_gt_avg_valid.png)

## 2. FairFace

In [8]:
!pip install datasets

In [ ]:
!python fairFace_dataset.py 

Téléchargement du dataset FairFace depuis HuggingFace...
README.md: 5.89kB [00:00, 5.80MB/s]
0.25/train-00000-of-00002-d405faba4f4b9b(…): 100%|████████████████████████████████████████████████████████████████████████████████████████████| 250M/250M [00:06<00:00, 38.9MB/s]
0.25/train-00001-of-00002-dd3cb681647274(…): 100%|████████████████████████████████████████████████████████████████████████████████████████████| 250M/250M [00:06<00:00, 41.7MB/s]
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
0.25/validation-00000-of-00001-951dbd63c(…): 100%|██████████████████████████████████████████████████████████████████████████████████████████| 63.2M/63.2M [00:03<00:00, 21.0MB/s]
Generating train split: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 86744/86744 [00:01<00:00, 77198.49 examples/s]
Generating validation split: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 10954/10954 [00:00<00:00, 81034.56 examples/s]
FairFace prêt !
Train: 68388, Valid: 14655, Test: 14655

In [ ]:
!python train.py --data_dir ./FairFace --tensorboard tf_log MODEL.METHOD dex

In [ ]:
!python test.py --data_dir ./FairFace --resume ./checkpoint/last.pth MODEL.METHOD dex

### Distribution des ages